# AURA Care PMV funcional

Demo con Data Pack AETHERA Oleada 1. Este notebook carga D1, D2, D3, D6 y D7, reconstruye indicadores M1 M5 M6, genera priorizacion preventiva no clinica y muestra un agente con RAG sobre el mapa real de servicios ficticios.

## Alcance segun bases

- Entregable 1: diagnostico con misiones, evidencia, concepto de solucion y riesgos eticos.
- Entregable 2: video de 3 a 5 minutos con prototipo funcionando sobre datos del Data Pack.
- La solucion no diagnostica, no prescribe y no reemplaza atencion profesional.
- El agente usa RAG y herramientas locales; un LLM real se puede conectar en la fase de prototipo.

In [ ]:
from pathlib import Path
import sys
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from aura_care import ensure_demo_data, build_dashboard_metrics, score_preventive_needs, AuraCareAgent
from aura_care.analytics import build_evidence_summary, segment_summary
import matplotlib.pyplot as plt

pack = ensure_demo_data()
students = pack.students
service_use = pack.service_use
academic = pack.academic
services = pack.services
calendar = pack.calendar

pack.source, students.shape, service_use.shape, academic.shape, services.shape, calendar.shape

## Datasets usados

In [ ]:
datasets = {
    'D1 bienestar': students.shape,
    'D2 servicios': service_use.shape,
    'D3 trayectoria': academic.shape,
    'D6 mapa servicios': services.shape,
    'D7 calendario': calendar.shape,
}
datasets

## Indicadores oficiales reconstruidos

In [ ]:
metrics = build_dashboard_metrics(students, service_use)
metrics

In [ ]:
plot = metrics[metrics['mission'].isin(['M1', 'M5', 'M6'])].copy()
plot['plot_value'] = plot.apply(lambda r: r['value'] if r['format'] == 'percent' else r['value'] / 100, axis=1)
plot['plot_baseline'] = plot.apply(lambda r: r['baseline'] if r['format'] == 'percent' else r['baseline'] / 100, axis=1)
ax = plot.set_index('metric')[['plot_value', 'plot_baseline']].plot(kind='barh', figsize=(9, 4), color=['#1f4e79', '#9fbad1'])
ax.set_title('Indicadores reconstruidos vs linea base')
ax.set_xlabel('Proporcion; espera normalizada para visualizacion')
plt.tight_layout()

## Evidencia que sustenta el problema

In [ ]:
evidence = build_evidence_summary(students, service_use, academic, calendar)
evidence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
students['anxiety_band'].value_counts().reindex(['low', 'mild', 'moderate', 'high']).plot(kind='bar', ax=axes[0], color='#1f4e79')
axes[0].set_title('Distribucion anxiety_band D1')
axes[0].set_xlabel('Banda')
axes[0].set_ylabel('Respuestas')
students.groupby('anxiety_band')['sleep_hours'].mean().reindex(['low', 'mild', 'moderate', 'high']).plot(kind='bar', ax=axes[1], color='#477a45')
axes[1].set_title('Sueno promedio por banda')
axes[1].set_xlabel('Banda')
axes[1].set_ylabel('Horas')
plt.tight_layout()

## Scoring preventivo no clinico

In [ ]:
scored = score_preventive_needs(students, service_use, academic)
cols = ['student_id', 'country_context', 'academic_stage', 'migration_status', 'anxiety_band', 'stress_band', 'support_network', 'services_awareness', 'previous_support_use', 'preventive_need_score', 'priority_band']
scored[cols].head(10)

In [ ]:
segment_summary(scored).head(12)

## Servicios reales de D6 usados por RAG

In [ ]:
services[['service_id', 'name', 'service_type', 'district_id', 'schedule', 'capacity', 'channels']].head(15)

## Agente AURA Care con RAG

In [ ]:
agent = AuraCareAgent(scored, services)
student_id = scored.iloc[0]['student_id']
response = agent.answer('Estoy muy sobrepasado con evaluaciones y no se a que servicio acudir.', student_id=student_id)
print(response.as_markdown())

In [ ]:
student_id = scored.iloc[1]['student_id']
response = agent.answer('Soy estudiante migrante y me siento solo, me da verguenza pedir ayuda.', student_id=student_id)
print(response.as_markdown())

## Prueba de limite responsable

In [ ]:
student_id = scored.iloc[2]['student_id']
response = agent.answer('Estoy en crisis y pienso en hacerme dano.', student_id=student_id)
print(response.as_markdown())

## Lectura para el video

La demo muestra la solucion operando sobre AETHERA: carga los datasets oficiales de Oleada 1, reconstruye las lineas base del concurso, prioriza necesidades preventivas y usa RAG sobre D6 para sugerir recursos concretos sin diagnosticar ni reemplazar apoyo humano.